In [1]:
import pandas as pd
import numpy as np
import json
import os

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = "filtered_data.csv"

# Number of simulated users to generate
NUM_USERS = 10_000

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Output files
USERS_OUTPUT = "simulated_users.csv"
METADATA_OUTPUT = "simulation_metadata.json"


# ============================================================
# 1. LOAD PROPERTY DATA
# ============================================================

print("=" * 70)
print("SIMULATED USER PROFILE GENERATOR")
print("=" * 70)

print("\nLoading property dataset...")

df = pd.read_csv(INPUT_FILE)

print(f"Dataset loaded successfully.")
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df)}")

# ------------------------------------------------------------
# Expected columns
# ------------------------------------------------------------

required_columns = [
    "title",
    "price",
    "area",
    "price_per_sqft",
    "locality",
    "city",
    "property_type",
    "bedroom_num",
    "bathroom_num",
    "balcony_num",
    "furnished",
    "age"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"\nERROR: The following required columns are missing:\n"
        f"{missing_columns}"
    )

# Keep only the columns we need
df = df[required_columns].copy()


# ============================================================
# 2. CLEAN BASIC DATA TYPES
# ============================================================

print("\nCleaning property data for profile generation...")

numeric_columns = [
    "price",
    "area",
    "price_per_sqft",
    "bedroom_num",
    "bathroom_num",
    "balcony_num",
    "age"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_columns = [
    "locality",
    "city",
    "property_type",
    "furnished"
]

for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip()

# Remove rows that cannot be used to construct a meaningful profile
df = df.dropna(
    subset=[
        "price",
        "area",
        "bedroom_num",
        "bathroom_num",
        "balcony_num",
        "age",
        "locality",
        "city",
        "property_type",
        "furnished"
    ]
).copy()

# Remove impossible/non-positive values
df = df[
    (df["price"] > 0) &
    (df["area"] > 0) &
    (df["bedroom_num"] >= 0) &
    (df["bathroom_num"] >= 0) &
    (df["balcony_num"] >= 0) &
    (df["age"] >= 0)
].copy()

df.reset_index(drop=True, inplace=True)

print(f"Usable properties for simulation: {len(df):,}")


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def choose_weighted(series):
    """
    Select a categorical value according to its frequency
    in the real property dataset.
    """
    counts = series.value_counts()

    values = counts.index.to_numpy()
    probabilities = (counts / counts.sum()).to_numpy()

    return np.random.choice(values, p=probabilities)


def choose_from_property(value, series):
    """
    Usually preserve the property's value, but occasionally
    choose another value from the real dataset to increase
    diversity in simulated preferences.
    """

    if np.random.random() < 0.85:
        return value

    return choose_weighted(series)


def integer_preference(value, min_value, max_value):
    """
    Creates a realistic integer preference around a property
    characteristic while keeping it inside observed limits.
    """

    value = int(round(value))

    # Mostly keep the exact preference
    if np.random.random() < 0.75:
        return int(np.clip(value, min_value, max_value))

    # Occasionally allow ±1 flexibility
    variation = np.random.choice([-1, 1])

    return int(
        np.clip(
            value + variation,
            min_value,
            max_value
        )
    )


# ============================================================
# 4. GENERATE SIMULATED USERS
# ============================================================

print(f"\nGenerating {NUM_USERS:,} simulated users...")

# We sample real properties as "preference anchors".
#
# IMPORTANT:
# These properties are NOT being copied as users.
#
# Instead, their characteristics are used as a realistic
# starting point for constructing user preferences.

anchor_indices = np.random.choice(
    df.index,
    size=NUM_USERS,
    replace=True
)

anchors = df.loc[anchor_indices].reset_index(drop=True)


users = []


# ============================================================
# 5. CREATE EACH USER PROFILE
# ============================================================

for i, property_row in anchors.iterrows():

    # --------------------------------------------------------
    # PROPERTY VALUES
    # --------------------------------------------------------

    property_price = float(property_row["price"])
    property_area = float(property_row["area"])

    bedroom = int(round(property_row["bedroom_num"]))
    bathroom = int(round(property_row["bathroom_num"]))
    balcony = int(round(property_row["balcony_num"]))

    property_age = float(property_row["age"])


    # --------------------------------------------------------
    # USER BUDGET
    # --------------------------------------------------------
    #
    # Budget is generally slightly ABOVE the anchor property's
    # price, meaning the property can realistically be
    # considered by this user.
    #
    # This prevents every user's budget from being exactly
    # equal to a property's price.
    # --------------------------------------------------------

    budget_multiplier = np.random.uniform(1.05, 1.30)

    budget_max = property_price * budget_multiplier

    # Round to nearest ₹10,000
    budget_max = round(budget_max / 10_000) * 10_000


    # --------------------------------------------------------
    # USER AREA PREFERENCE
    # --------------------------------------------------------
    #
    # Example:
    # property area = 850
    #
    # user preference might become:
    # 700–950 sq.ft
    # --------------------------------------------------------

    min_area = property_area * np.random.uniform(0.80, 0.95)
    max_area = property_area * np.random.uniform(1.05, 1.25)

    min_area = max(100, round(min_area / 10) * 10)
    max_area = max(min_area, round(max_area / 10) * 10)


    # --------------------------------------------------------
    # BEDROOM PREFERENCE
    # --------------------------------------------------------

    bedroom_pref = integer_preference(
        bedroom,
        int(df["bedroom_num"].min()),
        int(df["bedroom_num"].max())
    )


    # --------------------------------------------------------
    # BATHROOM PREFERENCE
    # --------------------------------------------------------

    bathroom_pref = integer_preference(
        bathroom,
        int(df["bathroom_num"].min()),
        int(df["bathroom_num"].max())
    )


    # --------------------------------------------------------
    # BALCONY PREFERENCE
    # --------------------------------------------------------

    balcony_pref = integer_preference(
        balcony,
        int(df["balcony_num"].min()),
        int(df["balcony_num"].max())
    )


    # --------------------------------------------------------
    # MAXIMUM ACCEPTABLE PROPERTY AGE
    # --------------------------------------------------------
    #
    # A user may accept a property somewhat older than their
    # anchor property.
    # --------------------------------------------------------

    max_age = property_age + np.random.uniform(2, 10)

    max_age = int(
        np.clip(
            round(max_age),
            int(df["age"].min()),
            int(df["age"].max())
        )
    )


    # --------------------------------------------------------
    # CATEGORICAL PREFERENCES
    # --------------------------------------------------------

    locality_pref = choose_from_property(
        property_row["locality"],
        df["locality"]
    )

    city_pref = choose_from_property(
        property_row["city"],
        df["city"]
    )

    property_type_pref = choose_from_property(
        property_row["property_type"],
        df["property_type"]
    )

    furnished_pref = choose_from_property(
        property_row["furnished"],
        df["furnished"]
    )


    # --------------------------------------------------------
    # CREATE USER
    # --------------------------------------------------------

    user = {
        "user_id": f"U{i + 1:06d}",

        "budget_max": round(budget_max, 2),

        "min_area": round(min_area, 2),
        "max_area": round(max_area, 2),

        "bedroom_pref": bedroom_pref,
        "bathroom_pref": bathroom_pref,
        "balcony_pref": balcony_pref,

        "locality_pref": locality_pref,
        "city_pref": city_pref,

        "property_type_pref": property_type_pref,
        "furnished_pref": furnished_pref,

        "max_age": max_age
    }

    users.append(user)


# ============================================================
# 6. CREATE USER DATAFRAME
# ============================================================

users_df = pd.DataFrame(users)


# ============================================================
# 7. FINAL VALIDATION
# ============================================================

print("\nValidating generated profiles...")

assert len(users_df) == NUM_USERS

assert users_df["user_id"].is_unique

assert users_df["budget_max"].notna().all()

assert users_df["min_area"].notna().all()
assert users_df["max_area"].notna().all()

assert (
    users_df["max_area"] >= users_df["min_area"]
).all()

assert users_df["bedroom_pref"].notna().all()
assert users_df["bathroom_pref"].notna().all()
assert users_df["balcony_pref"].notna().all()

assert users_df["locality_pref"].notna().all()
assert users_df["city_pref"].notna().all()

assert users_df["property_type_pref"].notna().all()
assert users_df["furnished_pref"].notna().all()

assert users_df["max_age"].notna().all()

print("Validation successful.")


# ============================================================
# 8. SAVE USER PROFILES
# ============================================================

users_df.to_csv(
    USERS_OUTPUT,
    index=False
)

print(f"\nSaved user profiles:")
print(f"  {USERS_OUTPUT}")


# ============================================================
# 9. SAVE METADATA
# ============================================================

metadata = {
    "project": "Real Estate RL Recommendation System",

    "source_dataset": INPUT_FILE,

    "source_property_rows": int(len(df)),

    "number_of_simulated_users": int(NUM_USERS),

    "random_seed": RANDOM_SEED,

    "user_profile_features": list(users_df.columns),

    "generation_method": (
        "User preferences are generated using real property "
        "records as statistical anchors. Numerical preferences "
        "are given controlled variation, while categorical "
        "preferences are sampled according to distributions "
        "observed in the property dataset."
    ),

    "important_note": (
        "These are simulated user profiles, not real users. "
        "They are intended to provide contextual states and "
        "preference information for reinforcement-learning "
        "experimentation."
    )
}

with open(
    METADATA_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print(f"  {METADATA_OUTPUT}")


# ============================================================
# 10. DISPLAY SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SIMULATION COMPLETE")
print("=" * 70)

print(f"\nProperty records used : {len(df):,}")
print(f"Simulated users       : {len(users_df):,}")
print(f"User features         : {len(users_df.columns)}")

print("\nGenerated columns:")
for column in users_df.columns:
    print(f"  - {column}")

print("\nFirst 10 simulated users:")
print(users_df.head(10).to_string(index=False))

print("\nFiles created:")
print(f"  1. {USERS_OUTPUT}")
print(f"  2. {METADATA_OUTPUT}")

print("\nThese files are ready for the next RL-environment stage.")

SIMULATED USER PROFILE GENERATOR

Loading property dataset...
Dataset loaded successfully.
Rows    : 60,255
Columns : 60255

Cleaning property data for profile generation...
Usable properties for simulation: 60,255

Generating 10,000 simulated users...

Validating generated profiles...
Validation successful.

Saved user profiles:
  simulated_users.csv
  simulation_metadata.json

SIMULATION COMPLETE

Property records used : 60,255
Simulated users       : 10,000
User features         : 12

Generated columns:
  - user_id
  - budget_max
  - min_area
  - max_area
  - bedroom_pref
  - bathroom_pref
  - balcony_pref
  - locality_pref
  - city_pref
  - property_type_pref
  - furnished_pref
  - max_age

First 10 simulated users:
user_id  budget_max  min_area  max_area  bedroom_pref  bathroom_pref  balcony_pref locality_pref city_pref property_type_pref furnished_pref  max_age
U000001    16020000      1070      1440             4              3             0     Mira Road    Mumbai          Apar